# RF Vpi Sweep — Scanning Fabry-Perot Cavity

This notebook runs a frequency / power sweep of an RF source while
recording Fabry-Perot cavity transmission traces on an oscilloscope.
The analysis extracts the modulation index $\beta$ from sideband / carrier
area ratios and fits $V_\pi$ at each RF frequency.

## How to use

1. **Install** both `hardwarelib` and `cavityscope` (editable installs).
2. **Edit the instrument section** below to match your lab setup (COM ports, VISA addresses, etc.).
3. **Adjust the sweep config** if needed (frequencies, powers, windows, …).
4. **Run all cells.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cavityscope.core import SweepConfig
from cavityscope.sweep import run_sweep

## 1. Instrument initialisation

Import **concrete** drivers from `hardwarelib` and configure them for your bench.
The sweep logic only sees the generic protocol interfaces, so you can swap
drivers freely.

In [ ]:
from hardwarelib.oscilloscopes.rigol import RigolDHO4000
from hardwarelib.signal_generators.windfreak import WindfreakSynthHD

SCOPE_RESOURCE  = "TCPIP0::192.168.1.50::INSTR"
SCOPE_CHANNEL   = 1
SCOPE_TIMEOUT   = 15_000        # ms

RF_COM_PORT     = "COM12"
RF_CHANNEL      = 0
RF_TIMEOUT      = 0.25          # s

scope = RigolDHO4000(SCOPE_RESOURCE, timeout_ms=SCOPE_TIMEOUT)
rf    = WindfreakSynthHD(RF_COM_PORT, channel=RF_CHANNEL, timeout_s=RF_TIMEOUT)

## 2. Sweep configuration

Adjust any parameter by passing it to `SweepConfig`.  
The defaults match the original script.

In [ ]:
cfg = SweepConfig(
    rf_frequencies_hz = np.arange(0.4e9, 5.1e9, 0.5e9).tolist(),
    rf_powers_dbm     = np.arange(-20.0, 10.1, 2.0).tolist(),

    cavity_fsr_hz         = 1.5e9,
    carrier_window_hz     = 120e6,
    sideband_window_hz    = 80e6,
    sideband_mode         = "both",

    compute_vpi           = True,
    net_power_offset_db   = 0.0,
    assumed_load_ohm      = 50.0,

    output_dir            = "vpi_sweep_output",
    save_trace_plots      = True,
    save_reference_plots  = True,
    save_frequency_plots  = True,
    save_raw_traces_csv   = True,

    # Optional: path to a power calibration CSV (columns: power_dbm, vpk_v
    # and optionally frequency_hz for per-frequency calibration).
    # When set, calibrated Vpk replaces the analytical dBm-to-V conversion.
    # power_calibration_csv = "path/to/calibration.csv",

    # Optional: live RF voltage measurement from a second scope channel.
    # Connect the RF output (before the modulator) to this channel with a
    # high-impedance probe. The measured Vpk overrides all other voltage
    # estimates — no interpolation, exact same moment & frequency.
    # live_cal_channel      = 2,
    # live_cal_cycles_for_rms = 20,
)

## 2b. Power calibration (optional)

If you have measured the actual RF voltage at the modulator for each power
setting, load a calibration to replace the analytical dBm→V conversion.

**Option A** — from a CSV file (set `power_calibration_csv` in the config above).  
**Option B** — build it from arrays right here.

In [ ]:
from cavityscope.core import PowerCalibration

# --- Option A: load from CSV (uncomment) ---
# cal = PowerCalibration.from_csv("path/to/calibration.csv")

# --- Option B: build from arrays (uncomment and fill in your data) ---
# cal = PowerCalibration.from_arrays(
#     power_dbm = np.array([-20, -10, 0, 10]),
#     vpk_v     = np.array([0.003, 0.010, 0.032, 0.100]),
#     # frequency_hz = np.array([...]),  # optional, for per-frequency cal
# )

# Set to None to use the default analytical conversion
cal = None

## 3. Run the sweep

In [ ]:
try:
    scope.open()
    rf.open()

    data = run_sweep(
        scope=scope,
        rf_source=rf,
        cfg=cfg,
        scope_channel=SCOPE_CHANNEL,
        calibration=cal,
    )
finally:
    try:
        rf.set_output(False)
    except Exception:
        pass
    rf.close()
    scope.close()

## 4. Inspect results

In [ ]:
df_results = data["results"]
df_fits    = data["fits"]
df_cal     = data.get("live_calibration", pd.DataFrame())

print(f"{len(df_results)} measurement points, {len(df_fits)} frequency fits")
if not df_cal.empty:
    print(f"{len(df_cal)} live calibration points (voltage source: live_scope)")
df_fits

In [ ]:
if not df_cal.empty:
    freqs = sorted(df_cal["rf_frequency_hz"].unique())
    fig, ax = plt.subplots(figsize=(9, 4))
    for freq in freqs:
        sub = df_cal[df_cal["rf_frequency_hz"] == freq].sort_values("rf_power_dbm")
        ax.plot(sub["rf_power_dbm"], sub["measured_vpk_v"], "o-", ms=4, lw=1.2,
                label=f"{freq/1e9:.2f} GHz")
    ax.set(xlabel="RF power setting (dBm)", ylabel="Measured Vpk (V)",
           title="Live calibration: measured RF voltage at modulator input")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7, ncol=max(1, len(freqs)//5))
    plt.tight_layout()
    plt.show()
else:
    print("No live calibration data (live_cal_channel not set).")

In [ ]:
from cavityscope.analysis.plotting import plot_vpi_vs_frequency

if not df_fits.empty:
    freq_ghz = df_fits["rf_frequency_hz"] / 1e9
    vpi = df_fits["fit_vpi_v"]
    r2 = df_fits["fit_r2"]
    n_pts = df_fits["n_fit_points"]
    valid = np.isfinite(vpi)

    fig, (ax_vpi, ax_r2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

    ax_vpi.plot(freq_ghz[valid], vpi[valid], "o-", markersize=7, lw=1.5)
    for f, v, n in zip(freq_ghz[valid], vpi[valid], n_pts[valid]):
        ax_vpi.annotate(f"N={int(n)}", (f, v), textcoords="offset points",
                        xytext=(0, 8), fontsize=7, ha="center", alpha=0.7)
    if valid.any():
        med = float(np.nanmedian(vpi[valid]))
        ax_vpi.axhline(med, ls="--", lw=0.8, color="gray", alpha=0.5,
                        label=f"median = {med:.3f} V")
    ax_vpi.set_ylabel("$V_\\pi$ (V)")
    ax_vpi.set_title("$V_\\pi$ vs RF frequency")
    ax_vpi.grid(True, alpha=0.25)
    ax_vpi.legend()

    bar_w = 0.8 * float(np.min(np.diff(freq_ghz))) if len(freq_ghz) > 1 else 0.1
    ax_r2.bar(freq_ghz[valid], r2[valid], width=bar_w, alpha=0.7)
    ax_r2.set_ylim(0, 1.05)
    ax_r2.axhline(0.99, ls=":", lw=0.8, color="gray", alpha=0.5, label="R²=0.99")
    ax_r2.set(xlabel="RF frequency (GHz)", ylabel="Fit R²")
    ax_r2.grid(True, alpha=0.25)
    ax_r2.legend()

    plt.tight_layout()
    plt.show()

## 5. Browse saved plots

Display all time-domain trace plots, frequency-space plots, and fit plots
that were saved during the sweep.

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage

run_dir = sorted(Path(cfg.output_dir).glob("measurement_*"))[-1]

def show_plots(folder: Path, heading: str):
    pngs = sorted(folder.glob("*.png"))
    if not pngs:
        return
    print(f"\n{'='*60}\n{heading}  ({len(pngs)} plots)\n{'='*60}")
    for p in pngs:
        display(IPImage(filename=str(p), width=900))

for name, label in [
    ("vpi_vs_frequency.png", "Vpi vs frequency summary"),
    ("live_calibration.png", "Live RF voltage calibration"),
]:
    p = run_dir / name
    if p.exists():
        print(f"\n{'='*60}\n{label}\n{'='*60}")
        display(IPImage(filename=str(p), width=900))

show_plots(run_dir / "reference_plots", "Reference traces (time domain)")
show_plots(run_dir / "trace_plots",     "RF-on traces (time domain)")
show_plots(run_dir / "frequency_plots", "Traces in frequency space")
show_plots(run_dir / "fit_plots",       "Per-frequency Vpi fits")